In [33]:
!pip install -q "google-genai>=2.12.1,<3.0" "google-auth==2.49.0" gradio

In [1]:
import gradio as gr
import time
from google import genai
from google.colab import userdata

import google.auth

print("Gradio version:", gr.__version__)
print("Google Auth version:", google.auth.__version__)
print("All libraries imported successfully!")

Gradio version: 6.32.0
Google Auth version: 2.49.0
All libraries imported successfully!


In [2]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [3]:
game_state = {
    "brass_key_found": False,
    "drawer_unlocked": False,
    "final_code_found": False,
    "escaped": False
}

print("New game session ready!")
print(game_state)

New game session ready!
{'brass_key_found': False, 'drawer_unlocked': False, 'final_code_found': False, 'escaped': False}


In [4]:
SYSTEM_PROMPT = """
You are the AI Game Master of The Mystery of the Locked Library.

Your responsibilities:
1. Describe the room and its objects.
2. Respond naturally to player actions.
3. Give clues when players investigate objects.
4. Track the player's progress.
5. Never reveal puzzle answers immediately.
6. Accept valid puzzle solutions.
7. Provide hints when requested.
8. Guide players toward the final exit.

GAME RULES:

- The brass key is hidden behind the painting.
- The desk drawer requires the brass key and code 2413.
- Inside the drawer, the player discovers code 7391.
- Code 7391 unlocks the main exit door.

IMPORTANT:
- Never reveal answers before discovery.
- Never display Python variables or internal game state.
- Keep responses immersive and natural.
- Never allow the player to escape without solving all puzzles.
- Follow the current game progress exactly.

Be engaging, logical and consistent.
"""

print("System prompt loaded successfully!")

System prompt loaded successfully!


In [5]:
conversation_history = []

def game_master(player_message):

    try:
        prompt = SYSTEM_PROMPT + "\n\n"

        prompt += "CURRENT GAME STATE:\n"
        prompt += str(game_state) + "\n\n"

        prompt += "PREVIOUS CONVERSATION:\n"

        for message in conversation_history:
            prompt += message + "\n"

        prompt += "\nPlayer: " + player_message
        prompt += "\nGame Master:"

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        answer = response.text

        conversation_history.append(
            "Player: " + player_message
        )

        conversation_history.append(
            "Game Master: " + answer
        )

        return answer

    except Exception as e:
        return "Game Master temporarily unavailable: " + str(e)


print("Gemini Game Master created successfully!")

Gemini Game Master created successfully!


In [6]:
test_response = game_master(
    "I am trapped in the library. What can I see around me?"
)

print(test_response)

You awaken to find yourself enclosed in a grand, wood-paneled library. The air is thick with the scent of aged paper and leather. Heavy velvet curtains are drawn tight over the tall windows, blocking out whatever lies beyond. 

The room is softly illuminated by a flickering brass desk lamp sitting in the center. To your left, a massive mahogany desk stands with a locked drawer. To your right, a towering bookshelf filled with leather-bound volumes stretches up to the shadowed ceiling. Above the marble fireplace across the room hangs a large, ornate oil painting of a stern-looking gentleman. And finally, dominating the far wall is a heavy oak exit door, secured with a complex digital keypad.

What would you like to do?


In [7]:
import gradio as gr

def respond(message, history):
    answer = game_master(message)
    return answer

demo = gr.ChatInterface(
    fn=respond,
    title="🔐 The Mystery of the Locked Library",
    description="Version 2 – AI Powered Escape Room. Investigate objects, discover clues and escape the library!"
)

print("Gradio interface created successfully!")

Gradio interface created successfully!


In [8]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://60b23f9635b0a152e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://60b23f9635b0a152e3.gradio.live


In [9]:
# Update game state after the successful first puzzle
game_state["brass_key_found"] = True

print("Puzzle 1 completed!")
print(game_state)

Puzzle 1 completed!
{'brass_key_found': True, 'drawer_unlocked': False, 'final_code_found': False, 'escaped': False}


In [13]:
def game_master(player_message):

    global conversation_history

    message = player_message.lower().strip()

    # PUZZLE 1: Find the brass key
    if not game_state["brass_key_found"]:

        if "painting" in message and any(
            word in message for word in
            ["examine", "inspect", "behind", "look", "move", "tilt"]
        ):

            game_state["brass_key_found"] = True

            return (
                "You carefully move the painting away from "
                "the wall. Behind its frame, you discover "
                "a small brass key! You take it."
            )

    # PUZZLE 2: Unlock the drawer
    if "2413" in message:

        if not game_state["brass_key_found"]:
            return "The drawer is locked. Find the brass key first."

        if not game_state["drawer_unlocked"]:

            game_state["drawer_unlocked"] = True

            return (
                "You insert the brass key and enter 2413. "
                "With a satisfying click, the drawer opens. "
                "Something appears to be hidden inside. "
                "What would you like to investigate?"
            )

    # PUZZLE 3: Discover the final code
    if game_state["drawer_unlocked"]:

        if not game_state["final_code_found"]:

            if any(word in message for word in
                   ["inside", "inspect", "paper", "note", "read"]):

                game_state["final_code_found"] = True

                return (
                    "Inside the drawer, you discover a small "
                    "slip of paper. Written on it is the final "
                    "exit code: 7391. You carefully take the note."
                )

    # PUZZLE 4: Unlock the exit
    if any(word in message for word in
           ["7391", "exit code", "door code"]):

        if not game_state["final_code_found"]:
            return (
                "You cannot unlock the exit yet. "
                "Investigate the drawer for the final clue."
            )

        if "7391" in message:

            game_state["escaped"] = True

            return (
                "Correct! The padlock opens with a satisfying "
                "click. The heavy oak doors swing open. "
                "Congratulations! You have escaped "
                "The Mystery of the Locked Library!"
            )

        return "That code is incorrect. Try again."

    # GENERAL AI RESPONSE
    prompt = SYSTEM_PROMPT + "\n\n"

    prompt += (
        "Current progress: "
        + str(game_state) + "\n\n"
    )

    prompt += (
        "Respond naturally to the player's action. "
        "Do not invent puzzle discoveries or reveal "
        "undiscovered puzzle answers."
    )

    for item in conversation_history:
        prompt += "\n" + item

    prompt += "\nPlayer: " + player_message
    prompt += "\nGame Master:"

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        answer = response.text

        conversation_history.append(
            "Player: " + player_message
        )

        conversation_history.append(
            "Game Master: " + answer
        )

        return answer

    except Exception as e:

        return "Game Master temporarily unavailable: " + str(e)


print("Improved Game Master ready!")

Improved Game Master ready!


In [11]:
test = game_master("I use the brass key and enter 2413 into the drawer.")

print(test)
print(game_state)

You walk over to the massive mahogany desk on the left side of the room. Inserting the gleaming brass key into the ornate keyhole of the locked drawer, you feel it turn with a smooth, satisfying click. 

However, the drawer remains stubbornly shut until you punch the digits **2-4-1-3** into the small mechanical dial built into the woodwork. With a soft pop, the latch gives way, and the drawer slides open.

Inside, nestled on a square of velvet, you discover a slip of paper with a freshly inked code written on it: **7391**, along with a note that reads, *"The final way forward."*

You carefully take the note with you. The desk drawer now sits half-open, and the heavy oak exit door across the room awaits your next move. What would you like to do?
{'brass_key_found': True, 'drawer_unlocked': True, 'final_code_found': True, 'escaped': False}


In [12]:
game_state = {
    "brass_key_found": True,
    "drawer_unlocked": False,
    "final_code_found": False,
    "escaped": False
}

conversation_history.clear()

print("Game state corrected!")
print(game_state)

Game state corrected!
{'brass_key_found': True, 'drawer_unlocked': False, 'final_code_found': False, 'escaped': False}


In [14]:
test = game_master(
    "I use the brass key and enter 2413 into the drawer."
)

print(test)
print(game_state)

You insert the brass key and enter 2413. With a satisfying click, the drawer opens. Something appears to be hidden inside. What would you like to investigate?
{'brass_key_found': True, 'drawer_unlocked': True, 'final_code_found': False, 'escaped': False}


In [15]:
test = game_master(
    "I inspect the inside of the drawer and read the paper."
)

print(test)
print(game_state)

Inside the drawer, you discover a small slip of paper. Written on it is the final exit code: 7391. You carefully take the note.
{'brass_key_found': True, 'drawer_unlocked': True, 'final_code_found': True, 'escaped': False}


In [16]:
test = game_master(
    "I enter 7391 into the exit keypad."
)

print(test)
print(game_state)


Correct! The padlock opens with a satisfying click. The heavy oak doors swing open. Congratulations! You have escaped The Mystery of the Locked Library!
{'brass_key_found': True, 'drawer_unlocked': True, 'final_code_found': True, 'escaped': True}


In [17]:
import gradio as gr

def respond(message, history):
    answer = game_master(message)
    return answer

demo = gr.ChatInterface(
    fn=respond,
    title="🔐 The Mystery of the Locked Library",
    description="Version 2 – AI Powered Escape Room. Investigate objects, discover clues and escape the library!"
)

In [18]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88901e6d55ea407a68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
import gradio as gr

def respond(message, history):
    return game_master(message)

demo = gr.ChatInterface(
    fn=respond,
    title="🔐 The Mystery of the Locked Library",
    description="AI Powered Escape Room – Find clues, solve puzzles and escape!"
)

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://afa817c03b99d40461.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
